# 07. Save and Load Models

Model progress can be saved during and after training. This means a model can resume where it left off and avoid long training times. Saving also means you can share your model and others can recreate your work. When publishing research models and techniques, most machine learning practitioners share:

* code to create the model, and
* the trained weights, or parameters, for the model

Sharing this data helps others understand how the model works and try it themselves with new data.

**Options**

There are different ways to save TensorFlow models depending on the API you're using. This guide uses tf.keras - a high-level API to build and train models in TensorFlow. The new, high-level `.keras` format used in this tutorial is recommended for saving Keras objects, as it provides robust, efficient name-based saving that is often easier to debug than low-level or legacy formats. For more advanced saving or serialization workflows, especially those involving custom objects, please refer to the Save and load Keras models guide. For other approaches, refer to the Using the SavedModel format guide.

### Setup

**Installs and imports**

Install and import TensorFlow and dependencies:

In [1]:
# pip install pyyaml h5py   # Required to save models in HDF5 format

In [3]:
import os

import tensorflow as tf
from tensorflow import keras

print(tf.version.VERSION)

2.21.0


**Get an example dataset**

To demonstrate how to save and load weights, you'll use the MNIST dataset. To speed up these runs, use the first 1000 examples:

In [4]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

train_labels = train_labels[:1000]
test_labels = test_labels[:1000]

train_images = train_images[:1000].reshape(-1, 28 * 28) / 255.0
test_images = test_images[:1000].reshape(-1, 28 * 28) / 255.0

**Define a model**

Start by building a simple sequential model:

In [5]:
# Define a simple sequential model
def create_model():
    model = tf.keras.Sequential([
        keras.layers.Dense(512, activation='relu', input_shape=(784,)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10)
    ])

    model.compile(optimizer='adam',
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])

    return model

# Create a basic model instance
model = create_model()

# Display the model's architecture
model.summary()

C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 512)                 │         401,920 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           5,130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 407,050 (1.55 MB)

 Trainable params: 407,050 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

### Save checkpoints during training

You can use a trained model without having to retrain it, or pick-up training where you left off in cae the training process was interrupted. The `tf.keras.callbacks.ModelCheckpoint` callback allows you to continually save the model both during and at the end of training.

**Checkpoint callback usage**

Create a `tf.keras.callbacks.ModelCheckpoint` callback that saves weights only during training:

In [7]:
# checkpoint_path = "training_1/cp.ckpt"
checkpoint_path = "training/cp_ckpt.weights.h5"
checkpoint_dir = os.path.dirname(checkpoint_path)

# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path, save_weights_only=True, verbose=1)

# Train the model with the new callback
model.fit(train_images, train_labels, epochs=10, validation_data=(test_images, test_labels), 
          callbacks=[cp_callback])   # Pass callback to training

# This may generate warnings related to saving the state of the optimizer
# These warnings (and similar warnings throughout this notebook)
# are in place to discourage outdated usage, and can be ignored.

Epoch 1/10
30/32 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 1.1735 - sparse_categorical_accuracy: 0.6708
Epoch 1: saving model to training/cp_ckpt.weights.h5

Epoch 1: finished saving model to training/cp_ckpt.weights.h5
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - loss: 1.1576 - sparse_categorical_accuracy: 0.6780 - val_loss: 0.7246 - val_sparse_categorical_accuracy: 0.7730
Epoch 2/10
31/32 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.4183 - sparse_categorical_accuracy: 0.8821
Epoch 2: saving model to training/cp_ckpt.weights.h5

Epoch 2: finished saving model to training/cp_ckpt.weights.h5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.4211 - sparse_categorical_accuracy: 0.8810 - val_loss: 0.5050 - val_sparse_categorical_accuracy: 0.8530
Epoch 3/10
29/32 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.2992 - sparse_categorical_accuracy: 0.9278
Epoch 3: saving model to training/cp_ckpt.weights.h5

Epoch 3: finished saving model to training/cp_ckpt.weights.h5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 27

This creates a single collection of TensorFlow checkpoint files that are updated at the end of each epoch:

In [8]:
os.listdir(checkpoint_dir)

['cp_ckpt.weights.h5']

As long as two models share the same architecture you can share weights between them. So, when restoring a model from weights-only, create a model with the same architecture as the original model and then set its weights.

Now rebuild a fresh, untrained model and evaluate it on the test set. An untrained model will perform at chance levels (~10% accuracy):

In [9]:
# Create a basic model instance
model = create_model()

# Evaluate the model
loss, acc = model.evaluate(test_images, test_labels, verbose=2)
print("Untrained model, accuracy: {:5.2f}%".format(100 * acc))

32/32 - 1s - 19ms/step - loss: 2.3386 - sparse_categorical_accuracy: 0.1090
Untrained model, accuracy: 10.90%


Then, load the weights from the checkpoint and re-evaluate:

In [10]:
# Load the weights
model.load_weights(checkpoint_path)

# Re-evaluate the model
loss, acc = model.evaluate(test_images, test_labels, verbose=2)
print("Restored model, accuracy: {:5.2f}%".format(100 * acc))

C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_1', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


32/32 - 0s - 9ms/step - loss: 0.3895 - sparse_categorical_accuracy: 0.8780
Restored model, accuracy: 87.80%


**Checkpoint callback options**

The callback provides several options to provide unique names for checkpoints and adjust the checkpointing frequency.

Train a new model, and save uniquely named checkpoints once every five epochs:

In [19]:
# Include the epoch in the file name (uses 'str.format')
# checkpoint_path = "training_2/cp-{epoch:04d}.ckpt"
checkpoint_path = "training_2/cp-{epoch:04d}-ckpt.weights.h5"
checkpoint_dir = os.path.dirname(checkpoint_path)

batch_size = 32

# Calculate the number of batches per epoch
import math
n_batches = len(train_images) / batch_size
n_batches = math.ceil(n_batches)   # Round up the number of batches to the nearest whole integer

# Create a callback that saves the model's weights every 5 epochs
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    verbose=1,
    save_weights_only=True,
    save_freq=5*n_batches)

# Create a new model instance
model = create_model()

# Save the weights using the 'checkpoint_path' format
# model.save_weights(checkpoint_path.format(epoch=0))

# Train the model with the new callback
model.fit(train_images, train_labels, epochs=50, batch_size=batch_size,
          callbacks=[cp_callback], validation_data=(test_images, test_labels), verbose=0)


Epoch 5: saving model to training_2/cp-0005-ckpt.weights.h5

Epoch 5: finished saving model to training_2/cp-0005-ckpt.weights.h5

Epoch 10: saving model to training_2/cp-0010-ckpt.weights.h5

Epoch 10: finished saving model to training_2/cp-0010-ckpt.weights.h5

Epoch 15: saving model to training_2/cp-0015-ckpt.weights.h5

Epoch 15: finished saving model to training_2/cp-0015-ckpt.weights.h5

Epoch 20: saving model to training_2/cp-0020-ckpt.weights.h5

Epoch 20: finished saving model to training_2/cp-0020-ckpt.weights.h5

Epoch 25: saving model to training_2/cp-0025-ckpt.weights.h5

Epoch 25: finished saving model to training_2/cp-0025-ckpt.weights.h5

Epoch 30: saving model to training_2/cp-0030-ckpt.weights.h5

Epoch 30: finished saving model to training_2/cp-0030-ckpt.weights.h5

Epoch 35: saving model to training_2/cp-0035-ckpt.weights.h5

Epoch 35: finished saving model to training_2/cp-0035-ckpt.weights.h5

Epoch 40: saving model to training_2/cp-0040-ckpt.weights.h5

Epoch 40

Now, review the resulting checkpoints and choose the latest one:

In [20]:
os.listdir(checkpoint_dir)

['cp-0005-ckpt.weights.h5',
 'cp-0010-ckpt.weights.h5',
 'cp-0015-ckpt.weights.h5',
 'cp-0020-ckpt.weights.h5',
 'cp-0025-ckpt.weights.h5',
 'cp-0030-ckpt.weights.h5',
 'cp-0035-ckpt.weights.h5',
 'cp-0040-ckpt.weights.h5',
 'cp-0045-ckpt.weights.h5',
 'cp-0050-ckpt.weights.h5']

In [31]:
# latest = tf.train.latest_checkpoint(checkpoint_dir)
# latest = tf.train.latest_checkpoint('training_2/')
# latest

To test, reset the model and load the latest checkpoint:

In [34]:
# Create a new model instance
model = create_model()

# Load the previously saved weights
# model.load_weights(latest)
model.load_weights('training_2/cp-0050-ckpt.weights.h5')

# Re-evaluate the model
loss, acc = model.evaluate(test_images, test_labels, verbose=2)
print("Restored model, accuracy: {:5.2f}%".format(100 * acc))

C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_12', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


32/32 - 1s - 20ms/step - loss: 0.4841 - sparse_categorical_accuracy: 0.8850
Restored model, accuracy: 88.50%


### What are these files?

The above code stores the weights to a collection of checkpoint-formatted files that contain only the trained weights in a binary format. Checkpoints contain:

* One or more shards that contain your model's weights.
* An index file that indicates which weights are stored in which shard.

If you are training a model on a single machine, you'll have one shard with the suffix: `.data-00000-of-00001`

### Manually save weights

To save weights manually, use `tf.keras.Model.save_weights`. By default, `tf.keras` - and the `Model.save_weights` method in particular - uses the TensorFlow Checkpoint format with a `.ckpt` extension. To save in the HDF5 format with a `.h5` extension, refer to the Save and load models guide.

In [37]:
# Save the weights
model.save_weights('my_checkpoint.weights.h5')

# Create a new model instance
model = create_model()

# Restore the weights
model.load_weights('my_checkpoint.weights.h5')

# Evaluate the model
loss, acc = model.evaluate(test_images, test_labels, verbose=2)
print("Restored model, accuracy: {:5.2f}%".format(100 * acc))

32/32 - 1s - 18ms/step - loss: 0.4841 - sparse_categorical_accuracy: 0.8850
Restored model, accuracy: 88.50%


### Save the entire model

Call `tf.keras.Model.save` to save a model's architecture, weights, and training configuraion in a single `model.keras` zip archive.

An entire model can be saved in three different file formats (the new `.keras` format and two legacy formats: `SavedModel`, and `HDF5`). Saving a model as `path/to/model.keras` automatically saves in the latest format.

**Note:** For Kera objects it's recommended to use the new high-level `.keras` format for richer, name-based saving and reloading, which is easier to debug. The low-level SavedModel format and legacy H5 format continue to be supported for existing code.

You can switch to the SavedModel format by:

* Passing `save_format='tf'` to `save()`
* Passing a filename without an extension

You can switch to the H5 format by:

* Passing `save_format='h5'` to `save()`
* Passing a filename that ends in `.h5`

Saving a fully-functional model is very useful - you can load them in TensorFlow.js (Saved Model, HDF5) and then train and run them in web browsers, or convert them to run on mobile devices using TensorFlow Lite (Saved Model, HDF5)

Custom objects (for example, subclassed models or layers) require special attention when saving and loading. Refer to the **Saving custom objects** section below.

**New high-level `.keras` format**

The new Keras v3 saving format, marked by the `.keras` extension, is a more simple, efficient format that implements name-based saving, ensuring what you load is exactly what you saved, from Python's perspective. This makes debugging much easier, and it is the recommended format for Keras.

The section below illustrates how to save and restore the model in the `.keras` format.

In [38]:
# Create and train a new model instance
model = create_model()
model.fit(train_images, train_labels, epochs=5)

# Save the entire model as a '.keras' zip archive
model.save('my_model.keras')

Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 1.2689 - sparse_categorical_accuracy: 0.6250
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.4583 - sparse_categorical_accuracy: 0.8710
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.3018 - sparse_categorical_accuracy: 0.9230
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.2203 - sparse_categorical_accuracy: 0.9440
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.1693 - sparse_categorical_accuracy: 0.9630


Reload a fresh Keras model from the `.keras`zip archive:

In [39]:
new_model = tf.keras.models.load_model('my_model.keras')

# Show the model architecture
new_model.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_28 (Dense)                     │ (None, 512)                 │         401,920 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_14 (Dropout)                 │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_29 (Dense)                     │ (None, 10)                  │           5,130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,221,152 (4.66 MB)

 Trainable params: 407,050 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 814,102 (3.11 MB)

Try running evaluate and predict with the loaded model:

In [40]:
# Evaluate the restored model
loss, acc = new_model.evaluate(test_images, test_labels, verbose=2)
print("Restored model, accuracy: {:5.2f}%".format(100 * acc))

print(new_model.predict(test_images).shape)

32/32 - 1s - 18ms/step - loss: 0.4124 - sparse_categorical_accuracy: 0.8660
Restored model, accuracy: 86.60%
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
(1000, 10)


**SavedModel format**

The SavedModel format is another way to serialize models. Models saved in this format can be restored using `tf.keras.models.load_model` and are compatible with TensorFlow Serving. The SavedModel guide goes into detail about how to `serve/inspect`the SavedModel. The section below illustrates the steps to save and restore the model.

In [42]:
# Create and traing a new model instance
model = create_model()
model.fit(train_images, train_labels, epochs=5)

# Save the entire model as a SavedModel
# !mkdir -p saved_model
model.save('saved_model/my_model', save_format='tf')

Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 1.1447 - sparse_categorical_accuracy: 0.6700
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.4174 - sparse_categorical_accuracy: 0.8890
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.2917 - sparse_categorical_accuracy: 0.9240
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.2081 - sparse_categorical_accuracy: 0.9520
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.1573 - sparse_categorical_accuracy: 0.9690


ValueError: The `save_format` argument is deprecated in Keras 3. Please remove this argument and pass a file path with either `.keras` or `.h5` extension.Received: save_format=tf

The SavedModel format is a directory containing a protobuf binary and a TensorFlow checkpoint. Inspect the saved model directory:

In [43]:
# my_model directory
# ls saved_model

# Contains an assets folaer, saved_model.pb, and variables folder.
# ls saved_model/my_model

Reload a fresh Keras model from the saved model:

In [44]:
new_model = tf.keras.models.load_model('saved_model/my_model')

# Check its architecture
new_model.summary()

ValueError: File format not supported: filepath=saved_model/my_model. Keras 3 only supports V3 `.keras` files, legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(saved_model/my_model, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).

The restored model is compiled with the same arguments as the original model. Try running evaluate and predict with the loaded model:

In [45]:
# Evaluate the restored model
# loss, acc = new_model.evaluate(test_images, test_labels, verbose=2)
# print("Restored model, accuracy: {:5.2f}%".format(100 * acc))

# print(new_model.predict(test_images).shape)

**HDF5 format**

Keras provides a basic legacy high-level save format using the HDF5 standard.

In [46]:
# Create and trainm a new model instance.
model = create_model()
model.fit(train_images, train_labels, epochs=5)

# Save the entire model to a HDF5 file.
# The '.h5' extension indicates that the model should be saved to HDF5.
model.save('my_model.h5')

Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 1.1864 - sparse_categorical_accuracy: 0.6660
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.4253 - sparse_categorical_accuracy: 0.8800
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.2874 - sparse_categorical_accuracy: 0.9220
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.2097 - sparse_categorical_accuracy: 0.9520
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.1605 - sparse_categorical_accuracy: 0.9590


Now, recreate the model from that file:

In [47]:
# Recreate the exact same model, including its weights and the optimizer
new_model = tf.keras.models.load_model('my_model.h5')

# Show the model architecture
new_model.summary()

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_34 (Dense)                     │ (None, 512)                 │         401,920 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_17 (Dropout)                 │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_35 (Dense)                     │ (None, 10)                  │           5,130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 407,052 (1.55 MB)

 Trainable params: 407,050 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

Check its accuracy:

In [49]:
loss, acc = new_model.evaluate(test_images, test_labels, verbose=2)
print("Restored model, accuracy: {:5.2f}%".format(100 * acc))

32/32 - 1s - 18ms/step - loss: 0.4239 - sparse_categorical_accuracy: 0.8700
Restored model, accuracy: 87.00%


Keras saves models by inspecting their architectures. This technique saves everything:

* The weight values
* The model's architecture
* The model's training configuration (what you pass to the `.compile()` method)
* The optimizer and its state, if any (this enables you to restart training where you left off)

Keras is not able to save the `v1.x` optimizers (from `tf.compat.v1.train`) since they aren't compatible with checkpoints. For v1.x optimizers, you need to re-compile the model after loading - losing the state of the optimizer.

**Saving custom objects**

If you are using the SavedModel format, you can skip this section. The key difference between high-level `.keras`/HDF5 formats and the low-level SavedModel format is that the `.keras`/HDF5 formats uses object configs to save the model architecture, while SavedModel saves the execution graph. Thus, SavedModels are able to save custom objects like subclassed models and custom layers without requiring the original code. However, debugging low-level SavedModel can be more difficult as a result, and we recommend using the high-level `.keras` format instead due to its name-based Keras-native nature.

To save custom objects to `.keras` and HDF5, you must do the following:

1. Define a `get_config` method in your object, and optionally a `from_config` classmethod.

* `get_config(self)` returns a JSON-serializable dictionary of parameters needed to recreate the object.
* `from_config(cls, config)` uses the returned config from `get_config` to create a new object. By default, this function will use the config as initialization kwargs (`return cls(**config)`).

2. Pass the custom objects to the model in one of three ways:

* Register the custom object with the `@tf.keras.utils.register_keras_serializable` decorator. (recommended)
* Directly pass the object to the `custom_objects` argument when loading the model. The argument must be a dictionary mapping the string class name to the Python class. E.g., `tf.keras.models.load_model(path, custom_objects={'CustomLayer': CustomLayer})
* Use a `tf.keras.utils.custom_object_scope` with the object included in the `custom_objects` dictionary argument, and place a `tf.keras.models.load_model(path)` call within the scope.

Refer to the Writing layers and models from scratch tutorial for examples of custom objects and `get_config`.